# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings so wide tables are readable
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

# Display settings so wide tables are readable
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

# Display settings so wide tables are readable
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

# Display settings so wide tables are readable
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

# Display settings so wide tables are readable
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
# The raw file contains a few Windows-1252 characters (e.g. en dashes), so it is read as latin-1
df = pd.read_csv('data/AviationData.csv', encoding='latin-1', low_memory=False)
print(f"Raw data: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Raw data: 88,889 rows x 31 columns


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,NC6404,Stinson,108-3,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,Fatal(4),Destroyed,NaN,N5069P,Piper,PA24-180,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,Fatal(3),Destroyed,NaN,N5142R,Cessna,172M,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,Fatal(2),Destroyed,NaN,N1168J,Rockwell,112,No,1.0,Reciprocating,NaN,NaN,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,Fatal(1),Destroyed,NaN,N15NY,Cessna,501,No,NaN,NaN,NaN,NaN,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [4]:
# Share of missing values in each column, highest first
missing = df.isna().mean().sort_values(ascending=False).round(3)
missing.to_frame('Fraction Missing')

,Fraction Missing
Schedule,0.858
Air.carrier,0.813
FAR.Description,0.640
Aircraft.Category,0.637
Longitude,0.613
Latitude,0.613
Airport.Code,0.436
Airport.Name,0.407
Broad.phase.of.flight,0.306
Publication.Date,0.155


In [5]:
# Summary statistics for the numeric columns
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


**Initial observations**

- There are about 88.9k events and 31 columns. Every column except the injury totals, engine count and coordinates is stored as text, so `Event.Date` must be converted to a datetime.
- Several columns are mostly empty: `Schedule`, `Air.carrier`, `FAR.Description`, `Aircraft.Category`, `Latitude` and `Longitude` are each more than 60% missing.
- The injury columns have 6–14% missing values, and the maximum values (hundreds of people) come from large airliner accidents, so those columns are very right-skewed.
- Categorical columns use several codes for "unknown" (`NaN`, `UNK`, `Unk`, `Unknown`), and `Make` is spelled inconsistently (`Cessna` vs `CESSNA`). These are handled below.

df.info()

# Share of missing values in each column, highest first
missing = df.isna().mean().sort_values(ascending=False).round(3)
missing.to_frame('Fraction Missing')

# Summary statistics for the numeric columns
df.describe()

In [6]:
# Convert the event date and look at the distribution of events by year
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df['Year'] = df['Event.Date'].dt.year
df['Year'].value_counts().sort_index().head(12)

Year
1948       1
1962       1
1974       1
1977       1
1979       2
1981       1
1982    3593
1983    3556
1984    3457
1985    3096
1986    2880
1987    2828
Name: count, dtype: int64

In [7]:
# Columns that decide whether an event is in scope for the client
for col in ['Amateur.Built', 'Aircraft.Category', 'Investigation.Type']:
    print(df[col].value_counts(dropna=False), '\n')

Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64 

Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64 

Investigation.Type
Accident    85015
Incident     3874
Name: count, dtype: int64 



**Filtering decisions**

1. **Date:** the client assumes a maximum 40-year lifetime for a make/model, so only events from **1983 onwards** are kept. Records before 1982 are very sparse anyway (single digits per year).
2. **Professional builds:** only aircraft with `Amateur.Built == "No"` are kept. The 102 records with a missing value are dropped, because it can't be confirmed that they are professional builds.
3. **Airplanes only:** `Aircraft.Category` is missing for roughly 64% of rows, so filtering on it directly would throw away most of the data. **Imputation:** a make almost always builds one kind of aircraft (Cessna and Piper build airplanes, Robinson builds helicopters). Missing categories are filled in this order:
   - the most common known category for the same **make and model**,
   - then, if that model never has a category recorded, the most common known category for the same **make**.

   After imputation only `Airplane` rows are kept, and a handful of known helicopter models from mainly-airplane makes (McDonnell Douglas/Hughes 369, 500, 520 and similar) are removed. Rows that still have no category (makes that never have one recorded) are dropped, since they can't be confirmed as airplanes.

In [8]:
# 1. Date filter
df = df[df['Year'] >= 1983]

# 2. Professional builds only
df = df[df['Amateur.Built'] == 'No']
print(f"After date and professional-build filters: {len(df):,} rows")

After date and professional-build filters: 76,960 rows


In [9]:
# Simple normalization used for the grouping keys during imputation
make_key = df['Make'].str.strip().str.upper()
model_key = df['Model'].str.strip().str.upper()

def most_common(series):
    """Most frequent non-null value of a series, or NaN if none exist."""
    counts = series.dropna().value_counts()
    return counts.index[0] if len(counts) else np.nan

model_mode = df.groupby([make_key, model_key])['Aircraft.Category'].agg(most_common)
make_mode = df.groupby(make_key)['Aircraft.Category'].agg(most_common)

was_missing = df['Aircraft.Category'].isna()
model_fill = pd.Series(list(zip(make_key, model_key)), index=df.index).map(model_mode)
make_fill = make_key.map(make_mode)

df['Aircraft.Category'] = df['Aircraft.Category'].fillna(model_fill).fillna(make_fill)
print(f"Categories imputed: {was_missing.sum() - df['Aircraft.Category'].isna().sum():,}")
print(f"Still missing after imputation: {df['Aircraft.Category'].isna().sum():,}")
df['Aircraft.Category'].value_counts(dropna=False).head(10)

Categories imputed: 50,730
Still missing after imputation: 825


Aircraft.Category
Airplane             66365
Helicopter            7836
Glider                1081
NaN                    825
Balloon                533
Weight-Shift           140
Powered Parachute       87
Gyrocraft               40
Ultralight              18
Unknown                 11
Name: count, dtype: int64

In [10]:
# 3. Keep airplanes only
df = df[df['Aircraft.Category'] == 'Airplane'].copy()

# A few helicopter models from makes that mostly build airplanes (McDonnell Douglas/Hughes 369, 500, 520,
# 530, 600N and 269 series) were imputed as "Airplane" from the make. Remove these known rotorcraft models.
rotorcraft = (df['Make'].str.contains('douglas|hughes|boeing', case=False, na=False) &
              df['Model'].str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
                .str.match(r'^(MD)?(369|500|520|530|600N|269|900|902|OH6|AH64)'))
print(f"Known rotorcraft models removed: {rotorcraft.sum()}")
df = df[~rotorcraft]
print(f"After keeping airplanes: {len(df):,} rows")

Known rotorcraft models removed: 32
After keeping airplanes: 66,333 rows


df.info()

# Share of missing values in each column, highest first
missing = df.isna().mean().sort_values(ascending=False).round(3)
missing.to_frame('Fraction Missing')

In [11]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
df[injury_cols + ['Injury.Severity']].isna().sum()

Total.Fatal.Injuries      8525
Total.Serious.Injuries    9446
Total.Minor.Injuries      8968
Total.Uninjured           4018
Injury.Severity            903
dtype: int64

In [12]:
# Rows where every injury column is missing carry no information about outcomes
all_missing = df[injury_cols].isna().all(axis=1)
print(f"Rows with all four injury counts missing: {all_missing.sum():,}")

# Injury.Severity records the fatality count as "Fatal(n)". Use it to fill missing fatal counts
fatal_from_severity = pd.to_numeric(
    df['Injury.Severity'].str.extract(r'Fatal\((\d+)\)')[0], errors='coerce'
)
filled_fatal = df['Total.Fatal.Injuries'].isna() & fatal_from_severity.notna()
df.loc[filled_fatal, 'Total.Fatal.Injuries'] = fatal_from_severity[filled_fatal]
print(f"Fatal counts recovered from Injury.Severity: {filled_fatal.sum():,} (in this subset, rows missing a fatal count never list one in Injury.Severity, so 0 is expected)")

# Drop rows with no injury information at all
df = df[~df[injury_cols].isna().all(axis=1)].copy()

# For the remaining rows, a missing count means none was reported in that category, so fill with 0
df[injury_cols] = df[injury_cols].fillna(0)

Rows with all four injury counts missing: 152
Fatal counts recovered from Injury.Severity: 0 (in this subset, rows missing a fatal count never list one in Injury.Severity, so 0 is expected)


In [13]:
# Estimated people on board = everyone accounted for in the four injury categories
df['Total.Passengers'] = df[injury_cols].sum(axis=1)

# Events with 0 people accounted for can't produce a rate, so they are removed
print(f"Rows with 0 people on board recorded: {(df['Total.Passengers'] == 0).sum():,}")
df = df[df['Total.Passengers'] > 0].copy()

# Key metric 1: share of people on board who were fatally or seriously injured
df['Fatal.Serious.Injuries'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']
df['Fatal.Serious.Fraction'] = df['Fatal.Serious.Injuries'] / df['Total.Passengers']

df[['Total.Passengers', 'Fatal.Serious.Injuries', 'Fatal.Serious.Fraction']].describe()

Rows with 0 people on board recorded: 973


,Total.Passengers,Fatal.Serious.Injuries,Fatal.Serious.Fraction
count,65208.000000,65208.000000,65208.000000
mean,7.471046,0.862502,0.269022
std,31.945171,6.250482,0.428010
min,1.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000
75%,3.000000,1.000000,0.666667
max,699.000000,349.000000,1.000000


**Derived injury columns**

- `Total.Passengers`: the estimated number of people on board, the sum of fatal, serious, minor and uninjured. The dataset has no seat or passenger count column, so this is the best available estimate. It includes crew.
- `Fatal.Serious.Injuries`: fatal plus serious injuries.
- `Fatal.Serious.Fraction`: `Fatal.Serious.Injuries / Total.Passengers`, the estimated **likelihood that a person on board is fatally or seriously injured** in an accident. It ranges from 0 to 1.

**Assumptions:** missing fatal counts are recovered from `Injury.Severity` where possible. Any other missing injury count is treated as 0, as long as at least one injury column was reported for that event. Rows where nothing was reported, or where 0 people are accounted for, are dropped because no rate can be computed.

# Convert the event date and look at the distribution of events by year
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df['Year'] = df['Event.Date'].dt.year
df['Year'].value_counts().sort_index().head(12)

In [14]:
df['Aircraft.damage'].value_counts(dropna=False)

Aircraft.damage
Substantial    47858
Destroyed      13252
NaN             2020
Minor           2004
Unknown           74
Name: count, dtype: int64

In [15]:
# Missing and "Unknown" damage can't be classified, so those rows are dropped
df = df[df['Aircraft.damage'].isin(['Destroyed', 'Substantial', 'Minor'])].copy()

# Key metric 2: 1 if the aircraft was destroyed, 0 otherwise. Averaging it gives the destruction rate
df['Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)
df['Destroyed'].value_counts()

Destroyed
0    49862
1    13252
Name: count, dtype: int64

**Aircraft damage:** the few rows with a missing or `Unknown` damage level are removed. The derived binary column `Destroyed` is 1 when the aircraft was destroyed and 0 for substantial or minor damage. Its mean for any group is that group's **rate of total destruction**.

**Filtering decisions**

1. **Date:** the client assumes a maximum 40-year lifetime for a make/model, so only events from **1983 onwards** are kept. Records before 1982 are very sparse anyway (single digits per year).
2. **Professional builds:** only aircraft with `Amateur.Built == "No"` are kept. The 102 records with a missing value are dropped, because it can't be confirmed that they are professional builds.
3. **Airplanes only:** `Aircraft.Category` is missing for roughly 64% of rows, so filtering on it directly would throw away most of the data. **Imputation:** a make almost always builds one kind of aircraft (Cessna and Piper build airplanes, Robinson builds helicopters). Missing categories are filled in this order:
   - the most common known category for the same **make and model**,
   - then, if that model never has a category recorded, the most common known category for the same **make**.

   After imputation only `Airplane` rows are kept, and a handful of known helicopter models from mainly-airplane makes (McDonnell Douglas/Hughes 369, 500, 520 and similar) are removed. Rows that still have no category (makes that never have one recorded) are dropped, since they can't be confirmed as airplanes.

In [16]:
# How messy is Make?
print(f"Distinct raw Make values: {df['Make'].nunique():,}")
df['Make'].value_counts().head(25)

Distinct raw Make values: 1,351


Make
Cessna               20671
Piper                11183
CESSNA                4717
Beech                 4000
PIPER                 2753
Mooney                1024
BEECH                 1001
Grumman                979
Boeing                 918
Bellanca               816
Air Tractor            576
Aeronca                455
Maule                  425
Champion               411
BOEING                 378
Aero Commander         338
De Havilland           333
Stinson                330
Rockwell               312
Taylorcraft            303
Luscombe               294
North American         290
Mcdonnell Douglas      261
MOONEY                 232
AIR TRACTOR INC        216
Name: count, dtype: int64

In [17]:
# Variants of the same manufacturer show up under different spellings and suffixes
for name in ['cessna', 'piper', 'beech', 'cirrus', 'havilland', 'learjet', 'embraer']:
    variants = df.loc[df['Make'].str.contains(name, case=False, na=False), 'Make'].value_counts()
    print(name.upper(), dict(variants.head(8)))

CESSNA {'Cessna': np.int64(20671), 'CESSNA': np.int64(4717), 'CESSNA AIRCRAFT CO': np.int64(24), 'CESSNA AIRCRAFT': np.int64(9), 'CESSNA AIRCRAFT COMPANY': np.int64(8), 'Cessna Ector': np.int64(3), 'CESSNA ECTOR': np.int64(3), 'Cessna Aircraft Company': np.int64(3)}
PIPER {'Piper': np.int64(11183), 'PIPER': np.int64(2753), 'PIPER AIRCRAFT INC': np.int64(25), 'PIPER AIRCRAFT CORPORATION': np.int64(7), 'NEW PIPER AIRCRAFT INC': np.int64(6), 'Piper/cub Crafters': np.int64(3), 'Piper Aircraft': np.int64(3), 'PIPER/CUB CRAFTERS': np.int64(3)}


BEECH {'Beech': np.int64(4000), 'BEECH': np.int64(1001), 'Beechcraft': np.int64(18), 'HAWKER BEECHCRAFT CORP': np.int64(13), 'Hawker Beechcraft Corp.': np.int64(7), 'Hawker Beechcraft': np.int64(7), 'HAWKER BEECHCRAFT': np.int64(7), 'BEECHCRAFT': np.int64(5)}
CIRRUS {'CIRRUS DESIGN CORP': np.int64(214), 'CIRRUS': np.int64(72), 'Cirrus': np.int64(68), 'Cirrus Design Corp.': np.int64(68), 'Cirrus Design Corporation': np.int64(10), 'Cirrus Design Corp': np.int64(5), 'Cirrus Design': np.int64(4), 'CIRRUS DESIGN CORP.': np.int64(4)}
HAVILLAND {'De Havilland': np.int64(333), 'DEHAVILLAND': np.int64(87), 'DE HAVILLAND': np.int64(23), 'de Havilland': np.int64(8), 'Dehavilland': np.int64(7), 'DeHavilland': np.int64(2), 'DEHAVILLAND CANADA': np.int64(1)}
LEARJET {'Learjet': np.int64(99), 'Gates Learjet': np.int64(57), 'LEARJET': np.int64(21), 'LEARJET INC': np.int64(9), 'GATES LEARJET CORP.': np.int64(7), 'Learjet Inc': np.int64(1), 'Gates Learjet Corporation': np.int64(1), 'BOMBARDIER LEARJET C

EMBRAER {'Embraer': np.int64(72), 'EMBRAER': np.int64(72), 'EMBRAER-EMPRESA BRASILEIRA DE': np.int64(3), 'EMBRAER S A': np.int64(3), 'Embraer Aircraft': np.int64(1), 'EMBRAER S.A.': np.int64(1), 'EMBRAER SA': np.int64(1), 'EMBRAER EXECUTIVE AIRCRAFT INC': np.int64(1)}


**Make cleaning tasks**

1. **Inconsistent capitalization and whitespace**: `Cessna` vs `CESSNA` vs `cessna `. Fix by converting to lowercase, stripping whitespace and then converting to title case.
2. **Punctuation and parenthetical notes**: for example `Ercoupe (Eng & Research Corp.)`. Remove anything in parentheses and any punctuation.
3. **Corporate suffixes**: `Inc`, `Corp`, `Co`, `Company`, `Aircraft`, `Industries`, `Design`, `GmbH` and so on. `Cirrus Design Corp.` and `Cirrus` are the same manufacturer, so remove these words.
4. **Known aliases**: Boeing-built Stearman biplanes (model 75 / PT-17 family) are moved to the `Boeing Stearman` make so they aren't mixed with Boeing airliners. Also map alternate names to one manufacturer (`Dehavilland` → `De Havilland`, `Gates Learjet` → `Learjet`, `Airbus Industrie` → `Airbus`, `Beechcraft`/`Hawker Beechcraft`/`Raytheon` → `Beech`, `Embraer S A` → `Embraer`, and others).
5. **Small makes**: keep only makes with at least **50** events after cleaning, so every make-level statistic rests on a reasonable sample.

In [18]:
import re

SUFFIXES = (r"\b(inc|corp|corporation|co|company|industries|industrie|ind|aircraft|acft|"
            r"aviation|avn|helicopters?|gmbh|ltd|limited|mfg|manufacturing|design|"
            r"international|intl|products|sa|s a|ag|ab|llc|airplane)\b")

MAKE_ALIASES = {
    'Dehavilland': 'De Havilland', 'Dehavilland Canada': 'De Havilland',
    'Gates Learjet': 'Learjet', 'Bombardier Learjet': 'Learjet',
    'Airbus Industrie': 'Airbus',
    'Beechcraft': 'Beech', 'Hawker Beechcraft': 'Beech', 'Hawker-Beechcraft': 'Beech',
    'Hawker Beech': 'Beech', 'Raytheon': 'Beech', 'Raytheon Corporate Jets': 'Beech',
    'Embraer-Empresa Brasileira De': 'Embraer', 'Embraer Executive': 'Embraer',
    'Mcdonnell-Douglas': 'Mcdonnell Douglas',
    'American Champion': 'Champion', 'American Champion Cor': 'Champion',
    'Grumman American': 'Grumman', 'Grumman-Schweizer': 'Grumman', 'Grumman Schweizer': 'Grumman',
    'Grumman Eng Cor-Schweizer': 'Grumman', 'Grumman Eng': 'Grumman',
    'Gulfstream American': 'Gulfstream', 'Gulfstream Aerospace': 'Gulfstream',
    'Gulfstream Aerospace Lp': 'Gulfstream',
    'Smith Ted Aerostar': 'Aerostar', 'Piper Aerostar': 'Aerostar',
    'North American Rockwell': 'Rockwell', 'Rockwell Commander': 'Rockwell',
    'Saab-Scania': 'Saab', 'Saab-Fairchild': 'Saab',
    'Fairchild Swearingen': 'Swearingen',
    'Short Bros': 'Short Brothers', 'Short Bros & Harland': 'Short Brothers',
    'Short Brothers & Harland': 'Short Brothers', 'Short Brothers Plc': 'Short Brothers',
    'British Aerospace Civil Aircr': 'British Aerospace', 'Bae': 'British Aerospace',
    'Burkhart Grob': 'Grob', 'Burkhart Grob Flugzeugbau': 'Grob', 'Grob-Werke': 'Grob',
    'Dassault-Breguet': 'Dassault', 'Avions Marcel Dassault': 'Dassault', 'Dassault Falcon': 'Dassault',
    'Cubcrafters': 'Cub Crafters', 'New Piper': 'Piper', 'Mooney Airplane': 'Mooney',
    'Flight': 'Flight Design', 'Eads Socata': 'Socata', 'Universal Stinson': 'Stinson', 'Ayres Thrush': 'Ayres',
}

def clean_make(make):
    """Normalize a raw Make string to a single manufacturer name."""
    if pd.isna(make):
        return np.nan
    m = make.lower()
    m = re.sub(r"\(.*?\)", " ", m)          # drop parenthetical notes
    m = re.sub(r"[^a-z0-9 \-&]", " ", m)     # drop punctuation
    m = re.sub(SUFFIXES, " ", m)              # drop corporate suffixes
    m = re.sub(r"\s+", " ", m).strip(" -&")
    m = m.title()
    return MAKE_ALIASES.get(m, m)

df['Make'] = df['Make'].map(clean_make)
df = df[df['Make'].notna() & (df['Make'] != '')]

# Stearman Kaydet biplanes (models 75, A75N1, PT-17, N2S...) were built by Boeing's Stearman division and are
# filed under both "Boeing" and "Boeing Stearman". Group them under one make so they are not mixed with airliners
stearman = (df['Make'] == 'Boeing') & df['Model'].str.upper().str.replace(r'[^A-Z0-9]', '', regex=True).str.match(
    r"^([A-E]?75(?!\d)|IB75|PT1[37]|N2S|STEARMAN)")
print(f"Boeing Stearman biplanes regrouped: {stearman.sum()}")
df.loc[stearman, 'Make'] = 'Boeing Stearman'
print(f"Distinct Make values after cleaning: {df['Make'].nunique():,}")
df['Make'].value_counts().head(40)

Boeing Stearman biplanes regrouped: 339
Distinct Make values after cleaning: 803


Make
Cessna               25434
Piper                13989
Beech                 5190
Grumman               1511
Mooney                1304
Bellanca               973
Boeing                 960
Air Tractor            890
Aeronca                605
Champion               602
Maule                  570
De Havilland           461
Cirrus                 449
Stinson                432
Rockwell               412
Aero Commander         405
Boeing Stearman        389
Luscombe               386
Taylorcraft            375
North American         366
Mcdonnell Douglas      307
Ayres                  276
Ercoupe                232
Aviat                  230
Douglas                211
Learjet                198
Diamond                173
Swearingen             156
Embraer                154
Gulfstream             151
Schweizer              140
Waco                   138
Lake                   138
Airbus                 138
Fairchild              137
Mitsubishi             130
Socata                 

In [19]:
# Keep makes with a reasonable number of events (at least 50)
MAKE_THRESHOLD = 50
make_counts = df['Make'].value_counts()
keep_makes = make_counts[make_counts >= MAKE_THRESHOLD].index
df = df[df['Make'].isin(keep_makes)].copy()
print(f"Makes kept: {len(keep_makes)}  |  rows remaining: {len(df):,}")
print(sorted(keep_makes))

Makes kept: 56  |  rows remaining: 60,253
['Aero Commander', 'Aeronca', 'Aerostar', 'Air Tractor', 'Airbus', 'American', 'Aviat', 'Ayres', 'Beech', 'Bellanca', 'Boeing', 'Boeing Stearman', 'Bombardier', 'British Aerospace', 'Canadair', 'Cessna', 'Champion', 'Christen', 'Cirrus', 'Cub Crafters', 'Dassault', 'De Havilland', 'Diamond', 'Douglas', 'Embraer', 'Ercoupe', 'Fairchild', 'Flight Design', 'Globe', 'Great Lakes', 'Grumman', 'Gulfstream', 'Helio', 'Lake', 'Learjet', 'Lockheed', 'Luscombe', 'Maule', 'Mcdonnell Douglas', 'Mitsubishi', 'Mooney', 'Navion', 'North American', 'Pilatus', 'Piper', 'Pitts', 'Rockwell', 'Ryan', 'Schweizer', 'Socata', 'Stinson', 'Swearingen', 'Taylorcraft', 'Waco', 'Weatherly', 'Wsk Pzl Mielec']


# Simple normalization used for the grouping keys during imputation
make_key = df['Make'].str.strip().str.upper()
model_key = df['Model'].str.strip().str.upper()

def most_common(series):
    """Most frequent non-null value of a series, or NaN if none exist."""
    counts = series.dropna().value_counts()
    return counts.index[0] if len(counts) else np.nan

model_mode = df.groupby([make_key, model_key])['Aircraft.Category'].agg(most_common)
make_mode = df.groupby(make_key)['Aircraft.Category'].agg(most_common)

was_missing = df['Aircraft.Category'].isna()
model_fill = pd.Series(list(zip(make_key, model_key)), index=df.index).map(model_mode)
make_fill = make_key.map(make_mode)

df['Aircraft.Category'] = df['Aircraft.Category'].fillna(model_fill).fillna(make_fill)
print(f"Categories imputed: {was_missing.sum() - df['Aircraft.Category'].isna().sum():,}")
print(f"Still missing after imputation: {df['Aircraft.Category'].isna().sum():,}")
df['Aircraft.Category'].value_counts(dropna=False).head(10)

In [20]:
print(f"Missing models: {df['Model'].isna().sum()}")
df = df.dropna(subset=['Model'])

# Normalize model labels: uppercase, remove spaces, and unify separators (e.g. "PA 28-140" -> "PA28-140")
df['Model'] = (df['Model'].str.upper()
                          .str.strip()
                          .str.replace(r'\s+', '', regex=True)
                          .str.replace(r'[_/]', '-', regex=True))
print(f"Distinct models: {df['Model'].nunique():,}")
df['Model'].value_counts().head(15)

Missing models: 13
Distinct models: 4,107


Model
152          2211
172          1614
172N         1084
PA-28-140     857
172M          781
150           742
172P          663
182           608
180           594
150M          550
PA-18-150     549
PA-18         548
PA-28-180     545
PA-28-161     523
PA-28-181     505
Name: count, dtype: int64

In [21]:
# Are model labels unique to a make? Count how many makes use each model label
makes_per_model = df.groupby('Model')['Make'].nunique().sort_values(ascending=False)
print(f"Model labels used by more than one make: {(makes_per_model > 1).sum()}")
shared = makes_per_model[makes_per_model > 1].head(10).index
df[df['Model'].isin(shared)].groupby(['Model', 'Make']).size().head(20)

Model labels used by more than one make: 225


Model  Make          
500    Aero Commander    17
       Cessna            26
       Embraer            1
       North American     1
       Rockwell           2
60     Beech             22
       Bombardier         1
       Learjet           11
690A   Aero Commander    16
       Gulfstream         1
       Rockwell          10
690B   Aero Commander     1
       Gulfstream         2
       Rockwell          18
700    Aero Commander     1
       Helio              1
       Rockwell           2
AA1A   American           3
       Grumman            2
       Gulfstream         3
dtype: int64

In [22]:
# Model labels are not unique (e.g. "35" is a Beech Bonanza and also used by other makes),
# so build a unique aircraft-type identifier from make + model
df['Make.Model'] = df['Make'] + ' ' + df['Model']
print(f"Distinct Make.Model identifiers: {df['Make.Model'].nunique():,}")
df['Make.Model'].value_counts().head(15)

Distinct Make.Model identifiers: 4,382


Make.Model
Cessna 152         2211
Cessna 172         1614
Cessna 172N        1084
Piper PA-28-140     857
Cessna 172M         781
Cessna 150          742
Cessna 172P         663
Cessna 182          608
Cessna 180          593
Cessna 150M         550
Piper PA-18-150     548
Piper PA-18         547
Piper PA-28-180     545
Piper PA-28-161     523
Piper PA-28-181     505
Name: count, dtype: int64

**Model:** rows with no model are dropped. Labels are uppercased and spaces are removed so that `PA 28` and `PA28` match. Several model labels (for example `35`, `65` or `T-34`) are shared by different manufacturers, so a model alone does not identify an aircraft type. The derived column **`Make.Model`** (for example `Cessna 172N`) is used as the unique identifier for a plane type.

df.info()

In [23]:
for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(f"--- {col}")
    print(df[col].value_counts(dropna=False), '\n')

--- Engine.Type
Engine.Type
Reciprocating      52161
NaN                 3088
Turbo Prop          2684
Turbo Fan           1399
Unknown              497
Turbo Jet            389
Turbo Shaft           20
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64 

--- Weather.Condition
Weather.Condition
VMC    52881
IMC     4825
NaN     1840
UNK      553
Unk      141
Name: count, dtype: int64 

--- Number.of.Engines
Number.of.Engines
1.0    48984
2.0     8187
NaN     2376
3.0      297
4.0      284
0.0      112
Name: count, dtype: int64 

--- Purpose.of.flight
Purpose.of.flight
Personal                     35218
Instructional                 8101
Unknown                       3904
Aerial Application            3441
Business                      2977
NaN                           2827
Positioning                   1037
Ferry                          562
Other Work Use                 522
Aerial Observation             381
Executive/corporate            376
Public Aircraft

In [24]:
# Engine.Type: unify the unknown codes and the rare turbofan variant
df['Engine.Type'] = df['Engine.Type'].replace({'UNK': 'Unknown', 'NONE': 'Unknown', 'LR': 'Unknown',
                                               'Geared Turbofan': 'Turbo Fan'})
df['Engine.Type'] = df['Engine.Type'].fillna('Unknown')

# Weather.Condition: UNK, Unk and NaN all mean unknown
df['Weather.Condition'] = df['Weather.Condition'].replace({'UNK': 'Unknown', 'Unk': 'Unknown'}).fillna('Unknown')

# Number.of.Engines: an airplane can't have 0 engines, so 0 is treated as missing
df.loc[df['Number.of.Engines'] == 0, 'Number.of.Engines'] = np.nan

# Purpose.of.flight: merge duplicate spellings and public-aircraft sub-types
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({
    'Air Race show': 'Air Race/show',
    'Public Aircraft - Federal': 'Public Aircraft',
    'Public Aircraft - Local': 'Public Aircraft',
    'Public Aircraft - State': 'Public Aircraft',
    'ASHO': 'Air Race/show',
    'PUBS': 'Public Aircraft', 'PUBL': 'Public Aircraft',
}).fillna('Unknown')

# Broad.phase.of.flight: NaN and the "Unknown" label are the same thing
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].fillna('Unknown')

for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(f"--- {col}")
    print(df[col].value_counts(dropna=False).head(12), '\n')

--- Engine.Type
Engine.Type
Reciprocating    52161
Unknown           3586
Turbo Prop        2684
Turbo Fan         1400
Turbo Jet          389
Turbo Shaft         20
Name: count, dtype: int64 

--- Weather.Condition
Weather.Condition
VMC        52881
IMC         4825
Unknown     2534
Name: count, dtype: int64 

--- Number.of.Engines
Number.of.Engines
1.0    48984
2.0     8187
NaN     2488
3.0      297
4.0      284
Name: count, dtype: int64 

--- Purpose.of.flight
Purpose.of.flight
Personal               35218
Instructional           8101
Unknown                 6731
Aerial Application      3441
Business                2977
Positioning             1037
Ferry                    562
Other Work Use           522
Public Aircraft          408
Aerial Observation       381
Executive/corporate      376
Skydiving                171
Name: count, dtype: int64 

--- Broad.phase.of.flight
Broad.phase.of.flight
Unknown        16764
Landing        12047
Takeoff         8984
Cruise          7153
Maneuv

**Cleaning applied to the other columns**

| Column | Issue | Fix |
|---|---|---|
| `Engine.Type` | `UNK`, `NONE`, `LR` and `NaN` are all unknown. `Geared Turbofan` has only a handful of rows | Merged into `Unknown`. Geared turbofan merged into `Turbo Fan` |
| `Weather.Condition` | `UNK`, `Unk` and `NaN` all mean unknown | Merged into `Unknown`. `VMC` = visual and `IMC` = instrument meteorological conditions |
| `Number.of.Engines` | `0` engines is impossible for an airplane | Set to `NaN`. Left missing rather than imputed |
| `Purpose.of.flight` | Duplicate spellings (`Air Race show` / `Air Race/show`) and public-aircraft sub-types | Merged. `NaN` → `Unknown` |
| `Broad.phase.of.flight` | About 30% missing, mostly in reports after 2008 | `NaN` → `Unknown`, so the rows stay in the data but are excluded when phase is analyzed |

injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
df[injury_cols + ['Injury.Severity']].isna().sum()

In [25]:
df.isna().mean().sort_values(ascending=False).round(3).head(15)

Schedule               0.878
Air.carrier            0.831
FAR.Description        0.706
Longitude              0.626
Latitude               0.626
Airport.Code           0.389
Airport.Name           0.360
Publication.Date       0.173
Report.Status          0.053
Number.of.Engines      0.041
Registration.Number    0.011
Country                0.002
Location               0.000
Event.Date             0.000
Accident.Number        0.000
dtype: float64

In [26]:
# Drop columns that are mostly empty (more than 50% missing) or that only identify the record
# and add nothing to the analysis
too_sparse = df.columns[df.isna().mean() > 0.5].tolist()
not_needed = ['Accident.Number', 'Registration.Number', 'Airport.Code', 'Airport.Name',
              'Amateur.Built', 'Aircraft.Category', 'Report.Status', 'Publication.Date']
to_drop = sorted(set(too_sparse + not_needed))
print("Dropping:", to_drop)
df = df.drop(columns=to_drop)
print(f"Final cleaned shape: {df.shape}")
df.info()

Dropping: ['Accident.Number', 'Air.carrier', 'Aircraft.Category', 'Airport.Code', 'Airport.Name', 'Amateur.Built', 'FAR.Description', 'Latitude', 'Longitude', 'Publication.Date', 'Registration.Number', 'Report.Status', 'Schedule']
Final cleaned shape: (60240, 24)
<class 'pandas.DataFrame'>
Index: 60240 entries, 3601 to 88886
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                60240 non-null  str           
 1   Investigation.Type      60240 non-null  str           
 2   Event.Date              60240 non-null  datetime64[us]
 3   Location                60212 non-null  str           
 4   Country                 60149 non-null  str           
 5   Injury.Severity         60240 non-null  str           
 6   Aircraft.damage         60240 non-null  str           
 7   Make                    60240 non-null  str           
 8   Model                   60240 non-n

**Columns removed:** columns that are more than 50% missing (such as `Latitude`, `Longitude`, `Schedule`, `Air.carrier` and `FAR.Description`), plus identifier or administrative columns that don't bear on safety (`Accident.Number`, `Registration.Number`, airport code and name, report status and publication date). `Amateur.Built` and `Aircraft.Category` are now constant after filtering, so they are dropped too.

# Estimated people on board = everyone accounted for in the four injury categories
df['Total.Passengers'] = df[injury_cols].sum(axis=1)

# Events with 0 people accounted for can't produce a rate, so they are removed
print(f"Rows with 0 people on board recorded: {(df['Total.Passengers'] == 0).sum():,}")
df = df[df['Total.Passengers'] > 0].copy()

# Key metric 1: share of people on board who were fatally or seriously injured
df['Fatal.Serious.Injuries'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']
df['Fatal.Serious.Fraction'] = df['Fatal.Serious.Injuries'] / df['Total.Passengers']

df[['Total.Passengers', 'Fatal.Serious.Injuries', 'Fatal.Serious.Fraction']].describe()

In [27]:
df.to_csv('data/AviationData_cleaned.csv', index=False)
print(f"Saved {len(df):,} cleaned rows to data/AviationData_cleaned.csv")

Saved 60,240 cleaned rows to data/AviationData_cleaned.csv


### Cleaning Summary

- **Scope:** professional-build airplanes with events from 1983 onwards. Aircraft category was imputed from make/model where missing.
- **Injury metric:** `Fatal.Serious.Fraction` is the estimated share of people on board who were fatally or seriously injured.
- **Damage metric:** `Destroyed` is a 0/1 flag whose mean gives the rate of total destruction.
- **Aircraft identity:** `Make` was normalized and kept only for makes with at least 50 events. `Make.Model` is a unique plane-type identifier.
- **Other factors:** engine type, weather, number of engines, purpose and phase of flight were standardized for the analysis notebook.